In [ ]:
N = 100   # or 50, 200, etc.

from datasets import load_dataset, Dataset, Audio

stream_train = load_dataset(
    "MLCommons/peoples_speech",
    name="clean",
    split="train",
    streaming=True,
)

# turn off decoding so it doesn't call torchcodec
stream_train = stream_train.cast_column("audio", Audio(decode=False))

small_stream = stream_train.take(N)
small_list = list(small_stream)          # now OK: only bytes/paths, no decode

small_ds = Dataset.from_list(small_list)
small_ds.save_to_disk(f"peoples_speech_small_{N}")


Resolving data files:   0%|          | 0/804 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/804 [00:00<?, ?it/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
from datasets import load_from_disk

N = 100
small_ds = load_from_disk(f"peoples_speech_small_{N}")

print(small_ds)
print(small_ds[0]["audio"].keys())  # will show "bytes"


Dataset({
    features: ['id', 'audio', 'duration_ms', 'text'],
    num_rows: 100
})
dict_keys(['bytes', 'path'])


In [ ]:
!pip install soundfile

import io
import soundfile as sf
import numpy as np

def decode_audio(batch):
    audio_bytes = batch["audio"]["bytes"]
    with io.BytesIO(audio_bytes) as f:
        waveform, sr = sf.read(f, dtype="float32")

    # Force numpy array
    waveform = np.asarray(waveform, dtype="float32")

    # If 2D (stereo), average channels → mono
    if waveform.ndim == 2:
        waveform = waveform.mean(axis=1)

    batch["audio_array"] = waveform
    batch["sampling_rate"] = int(sr)
    return batch

small_ds_decoded = small_ds.map(decode_audio)

# Force to numpy when printing, in case HF stored as list
arr0 = np.asarray(small_ds_decoded[0]["audio_array"], dtype="float32")
print(type(arr0), arr0.shape)
print(small_ds_decoded[0]["sampling_rate"])


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

<class 'numpy.ndarray'> (238720,)
16000


In [ ]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained("openai/whisper-small")

def prepare_batch(batch):
    out = processor(
        batch["audio_array"],          # numpy waveform
        sampling_rate=batch["sampling_rate"],
        text=batch["text"],            # existing transcript
        return_attention_mask=True,
    )
    batch["input_features"] = out["input_features"][0]
    batch["labels"] = out["labels"]
    return batch

small_ds_proc = small_ds_decoded.map(
    prepare_batch,
    remove_columns=["audio", "audio_array", "sampling_rate"],
)
print(small_ds_proc)


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'duration_ms', 'text', 'input_features', 'labels'],
    num_rows: 100
})


In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    feats = [torch.tensor(b["input_features"]) for b in batch]
    max_len = max(f.shape[0] for f in feats)
    padded = []
    for f in feats:
        pad_len = max_len - f.shape[0]
        if pad_len > 0:
            f = torch.nn.functional.pad(f, (0, 0, 0, pad_len))
        padded.append(f)
    input_features = torch.stack(padded)

    labels_list = [torch.tensor(b["labels"]) for b in batch]
    labels = pad_sequence(labels_list, batch_first=True, padding_value=-100)

    return {"input_features": input_features, "labels": labels}

train_dataloader = DataLoader(
    small_ds_proc,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn,
)


In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

model.train()
for step, batch in enumerate(train_dataloader):
    batch = {k: v.to(device) for k, v in batch.items()}
    optimizer.zero_grad()
    outputs = model(
        input_features=batch["input_features"],
        labels=batch["labels"],
    )
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    print(f"step {step} loss {loss.item():.4f}")
    if step == 5:
        break


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

step 0 loss 1.5124
step 1 loss 1.3853
step 2 loss 1.3888
step 3 loss 1.5731
step 4 loss 1.5086
step 5 loss 1.0258


In [18]:
from transformers import AutoProcessor

processor.save_pretrained("whisper_small_peoplespeech_100")
model.save_pretrained("whisper_small_peoplespeech_100")


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


In [19]:
from transformers import WhisperForConditionalGeneration, AutoProcessor
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained("whisper_small_peoplespeech_100", local_files_only=True)
model = WhisperForConditionalGeneration.from_pretrained("whisper_small_peoplespeech_100", local_files_only=True).to(device)
model.eval()


WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (f

In [20]:
ex = small_ds_decoded[0]  # or any index
inputs = processor(
    ex["audio_array"],
    sampling_rate=ex["sampling_rate"],
    return_tensors="pt",
)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    generated_ids = model.generate(input_features=inputs["input_features"])

pred = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("GT  :", ex["text"])
print("Pred:", pred)


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
`generation_config` default values have been modified to match model-specific defaults: {'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}. If this is not desired, please set these values explicitly.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transform

GT  : i wanted this to share a few things but i'm going to not share as much as i wanted to share because we are starting late i'd like to get this thing going so we all get home at a decent hour this this election is very important to
Pred:  I wanted to just share a few things, but I'm going to not share as much as I wanted to share because we are starting late. I'd like to get this thing going so we all get home at a decent hour. This election is very important to us.


In [32]:
!wget -O sample_speech.wav \
  https://raw.githubusercontent.com/Jakobovski/free-spoken-digit-dataset/master/recordings/7_jackson_43.wav

import soundfile as sf
import numpy as np
import librosa
import torch

filename = "/content/sample_speech.wav"

# 1. Load original audio
waveform, sr = sf.read(filename, dtype="float32")
waveform = np.asarray(waveform, dtype="float32")
if waveform.ndim == 2:
    waveform = waveform.mean(axis=1)

# 2. Resample to 16 kHz for Whisper
target_sr = 16000
if sr != target_sr:
    waveform = librosa.resample(waveform, orig_sr=sr, target_sr=target_sr)
    sr = target_sr

# 3. Run Whisper
inputs = processor(waveform, sampling_rate=sr, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    gen_ids = model.generate(input_features=inputs["input_features"])

pred = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
print("Transcript:", pred)


--2025-12-16 13:27:09--  https://raw.githubusercontent.com/Jakobovski/free-spoken-digit-dataset/master/recordings/7_jackson_43.wav
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8650 (8.4K) [audio/wav]
Saving to: ‘sample_speech.wav’

sample_speech.wav   100%[===================>]   8.45K  --.-KB/s    in 0s      

2025-12-16 13:27:09 (58.9 MB/s) - ‘sample_speech.wav’ saved [8650/8650]

Transcript:  7.


In [33]:
ex = small_ds_decoded[0]  # or any index

inputs = processor(
    ex["audio_array"],
    sampling_rate=ex["sampling_rate"],
    return_tensors="pt",
)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    gen_ids = model.generate(input_features=inputs["input_features"])

pred = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
print("GT  :", ex["text"])
print("Pred:", pred)


GT  : i wanted this to share a few things but i'm going to not share as much as i wanted to share because we are starting late i'd like to get this thing going so we all get home at a decent hour this this election is very important to
Pred:  I wanted to just share a few things, but I'm going to not share as much as I wanted to share because we are starting late. I'd like to get this thing going so we all get home at a decent hour. This election is very important to us.
